In [ ]:
import xarray as xr
import os   
import parflow as pf
import plotly.express as px
from parflow.tools.hydrology import calculate_overland_flow_grid, calculate_subsurface_storage, calculate_water_table_depth
import numpy as np
import shutil
import json
import plotly.io as pio

In [ ]:
root_dir = "/glade/derecho/scratch/bwest/drought-ensemble"
domain = "potomac_save"
ensemble_name = "pumping_ensemble"
ensemble_member = "pumping_0_01"
files = json.load(open(f"{root_dir}/domains/{domain}/processed_full_runs/{ensemble_name}/{ensemble_member}/file_locations.json"))
data = xr.open_mfdataset(files, concat_dim="time", combine="nested")
data.info()

/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'cfradial1' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'furuno' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'gamic' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
ERROR 1: PROJ: proj_create_from_database: Open of /glade/work/bwest/conda-envs/droughts/share/proj failed
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: 

xarray.Dataset {
dimensions:
	time = 157680 ;
	z = 10 ;
	y = 41 ;
	x = 78 ;

variables:
	float64 pressure(time, z, y, x) ;
	float64 saturation(time, z, y, x) ;
	float64 evaptrans(time, z, y, x) ;
	float64 overland_bc_flux(time, y, x) ;
	float64 mask(time, z, y, x) ;
	float64 mannings(time, y, x) ;
	float64 porosity(time, z, y, x) ;
	float64 specific_storage(time, z, y, x) ;
	float64 DZ_Multiplier(time, z, y, x) ;
	float64 slopex(time, y, x) ;
	float64 slopey(time, y, x) ;
	float64 perm_x(time, z, y, x) ;
	float64 perm_y(time, z, y, x) ;
	float64 perm_z(time, z, y, x) ;
	float64 overland_flow(time, y, x) ;
	float64 subsurface_storage(time, z, y, x) ;
	float64 time(time) ;

// global attributes:
}

In [ ]:
pio.templates.default = "plotly_white"
outlet_x = 18
outlet_y = 21
data["overland_flow"] = data["overland_flow"].isel(x=outlet_x, y=outlet_y)

In [ ]:
data["year"] = data.time 
fig = px.line(data.overland_flow.isel(time=slice(24, None, 24)))
# change the x axis to be in years

# change x and y axis labels
fig.update_xaxes(title="")
fig.update_yaxes(title="")
# remove x and y axis numbers
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

# remove background grid
fig.update_layout(
    xaxis=dict(
        showgrid=False,
        showline=True,
        linewidth=2,
        linecolor='black'
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,
        linewidth=2,

    )
)
# set the height of the figure to be 400
fig.update_layout(height=200, width=900)



In [ ]:
years = range(1, 18)
cumulative_streamflows = []
for year in years:
    total_streamflow = data.overland_flow.isel(time=slice(8760*(year-1), 8760*(year))).sum(dim="time")
    cumulative_streamflows.append(float(total_streamflow))

# cumulative_streamflows = xr.concat(cumulative_streamflows, dim="year")

# plot the cumulative streamflows
fig = px.bar(x=years, y=cumulative_streamflows)


In [ ]:
fig.update_xaxes(title="Year")
fig.update_yaxes(title="Cumulative Streamflow (m3)")
fig.show()

In [ ]:
cumulative_streamflows

[959093874.8152953,
 850804972.342071,
 849751931.7660002,
 809441475.5764582,
 783177179.9700358,
 778942187.0116931,
 776880211.3324957,
 775687966.5808337,
 774885894.9450761,
 774316788.1231208,
 773928283.3275663,
 773565403.1207324,
 773306102.2493618,
 773054183.3087724,
 772950025.1320808,
 772768412.1063395,
 772650834.1175197]

In [ ]:
streamflow_difference_from_previous_year = []
for year in years:
    if year == 0:
        continue
    streamflow_difference_from_previous_year.append(float((cumulative_streamflows[year-1] - cumulative_streamflows[year-2])/cumulative_streamflows[year-2]))

px.bar(x=years[4:11], y=streamflow_difference_from_previous_year[4:11])

